# CARec remaining experiments on Colab

Prepared from local checkpoints inspected on 2026-11-14. Run shared cells first, then choose individual model cells. Do not use Run All for the first check.

- Multilingual E5-large without instruction: phase 2 only, from existing phase-one weights.
- Multilingual E5-large with instruction: both phases.
- INSTRUCTOR-base without instruction: both phases.
- INSTRUCTOR-base with instruction: both phases.

MiniLM, MPNet, E5-base and both Qwen conditions already have both phase checkpoints and are omitted. Checkpoint presence establishes saved phases, not an independent quality audit.

## 1. Transfer inputs once

On your Mac, run this command in the NI-PML project terminal. It creates a transfer archive without running training:

```bash
COPYFILE_DISABLE=1 tar -czf /tmp/ni-pml-colab-inputs.tar.gz \
  data/processed/electronics_carec_035_seed2023/{train,valid,test,cold}.tsv.gz \
  data/processed/electronics_carec_035_seed2023/item_ids.tsv.gz \
  artifacts/embeddings/electronics_carec_035_seed2023/multilingual-e5-large-instruct-{with,without}_instruction.npz \
  artifacts/embeddings/electronics_carec_035_seed2023/instructor-base-{with,without}_instruction.npz \
  artifacts/carec/electronics_carec_035_seed2023/multilingual-e5-large-instruct-without_instruction/phase1-best.pt
```

Upload `/tmp/ni-pml-colab-inputs.tar.gz` to `My Drive/PML/`. Use your existing PML folder in My Drive. Keep the archive name unchanged.

Open this notebook in Colab, select a GPU runtime, or connect from VS Code with the official Colab extension. Local files and packages are not transferred by selecting a remote kernel. Drive mounting may require the extension's `Colab: Mount Google Drive to Server...` command or authorization in browser Colab.

## 2. Install the training environment

Run once per new runtime, then restart the kernel before continuing. Python 3.11 or 3.12 is required by this pinned environment. In browser Colab, choose Runtime > Change runtime type > Runtime Version > 2026.07, which uses Python 3.12, and select a GPU. Save and reconnect before installing. See [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html) and the [Python 3.13 upgrade notice](https://github.com/googlecolab/colabtools/issues/6081). The installer does not import NumPy or Torch before installation. No text encoders are downloaded.

In [ ]:
import sys
import subprocess

if sys.version_info[:2] not in ((3, 11), (3, 12)):
    raise RuntimeError("Use a Colab runtime with Python 3.11 or 3.12 for this environment.")
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "torch==2.6.0", "--index-url", "https://download.pytorch.org/whl/cu124",
])
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet",
    "numpy==1.26.4", "pandas==2.2.3", "scipy==1.14.1", "scikit-learn==1.6.1",
    "torch-geometric==2.6.1", "recbole==1.1.1", "tqdm==4.67.1",
])
print("Installation finished. Restart the kernel, then continue with section 3.")

## 3. Mount Drive and stage inputs

Inputs are extracted to temporary Colab storage for training. Only best phase weights are saved to Drive. Metrics remain in notebook outputs and memory. A phase-one checkpoint starts phase two with a fresh optimizer; weights-only checkpoints cannot resume an exact interrupted epoch.

In [ ]:
from pathlib import Path
import shutil
import tarfile
from google.colab import drive

drive.mount("/content/drive")
DRIVE_ROOT = Path("/content/drive/MyDrive/PML")
PROJECT_ROOT = Path("/content/ni-pml")
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)
archive_path = DRIVE_ROOT / "ni-pml-colab-inputs.tar.gz"
if not archive_path.is_file():
    raise FileNotFoundError(f"Upload the input archive to {archive_path}")
with tarfile.open(archive_path, "r:gz") as archive:
    # Accept only regular input files under the two expected directories.
    for member in archive.getmembers():
        relative = Path(member.name)
        if (not member.isfile() or relative.is_absolute() or ".." in relative.parts
                or relative.parts[0] not in {"data", "artifacts"}):
            raise ValueError(f"Unexpected archive entry: {member.name}")
        destination = PROJECT_ROOT / relative
        destination.parent.mkdir(parents=True, exist_ok=True)
        with archive.extractfile(member) as src, destination.open("wb") as dst:
            shutil.copyfileobj(src, dst)
print("Inputs staged at", PROJECT_ROOT)

## 4. Load pinned CARec and select CUDA

Uses the same upstream commit as the Mac notebook. Sparse normalized graph propagation, MLP and original uniformity loss all run on CUDA. No CPU graph bridge is used. CUDA throughput and numerical agreement still require a user-run check.

Defaults copy the current Mac notebook: 90 phase-one epochs, 30 phase-two epochs, patience 15, batch size 2048. Set both epoch limits to 1 before a first check. A check creates checkpoints, so use a separate `OUTPUT_GROUP`, such as `carec-colab-check`, then switch back and rerun shared settings for the full experiments.

In [ ]:
import subprocess
import sys
from copy import copy, deepcopy
from time import perf_counter

import numpy as np
import pandas as pd
import torch
from tqdm.auto import tqdm
from IPython.display import display

CAREC_COMMIT = "bbfc7f385e796840f6eb5c967dfc8714284958ce"
CAREC_ROOT = PROJECT_ROOT / "external/CARec"
if not (CAREC_ROOT / ".git").exists():
    CAREC_ROOT.parent.mkdir(parents=True, exist_ok=True)
    subprocess.check_call(["git", "clone", "https://github.com/ChenMetanoia/CARec.git", str(CAREC_ROOT)])
    subprocess.check_call(["git", "-C", str(CAREC_ROOT), "checkout", "--detach", CAREC_COMMIT])
revision = subprocess.check_output(["git", "-C", str(CAREC_ROOT), "rev-parse", "HEAD"], text=True).strip()
changes = subprocess.check_output([
    "git", "-C", str(CAREC_ROOT), "status", "--porcelain", "--untracked-files=no"
], text=True).strip()
if revision != CAREC_COMMIT or changes:
    raise RuntimeError("The CARec checkout differs from the pinned, unchanged source.")
for module_name in ("model", "utils", "recbole_gnn"):
    module = sys.modules.get(module_name)
    if module is not None:
        paths = list(getattr(module, "__path__", []))
        if not paths or any(not Path(p).resolve().is_relative_to(CAREC_ROOT) for p in paths):
            raise RuntimeError(f"Import conflict for {module_name}. Restart the kernel.")
if str(CAREC_ROOT) not in sys.path:
    sys.path.insert(0, str(CAREC_ROOT))
from types import SimpleNamespace
from model.collabcontex import CollabContex
carec = SimpleNamespace(CollabContex=CollabContex)
from recbole.config import Config
from recbole.data.dataset import Dataset
from recbole.data.interaction import Interaction
from recbole.data.dataloader import FullSortEvalDataLoader
from recbole.sampler import Sampler
from recbole.evaluator import Collector, Evaluator
from recbole.utils import init_seed
from recbole.utils.enum_type import FeatureSource, FeatureType

if not torch.cuda.is_available():
    raise RuntimeError("No CUDA GPU. Select a GPU Colab runtime and reconnect.")
if torch.__version__.split("+")[0] != "2.6.0" or np.__version__ != "1.26.4":
    raise RuntimeError("Restart the kernel after installing the pinned packages.")
device = torch.device("cuda:0")
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
DATASET_ID = "electronics_carec_035_seed2023"
PREPARED_DIR = PROJECT_ROOT / "data/processed" / DATASET_ID
RECBOLE_ROOT = PROJECT_ROOT / "artifacts/recbole"
RECBOLE_DIR = RECBOLE_ROOT / DATASET_ID
EMBEDDING_DIR = PROJECT_ROOT / "artifacts/embeddings" / DATASET_ID
OUTPUT_GROUP = "carec-colab"
RUNS_DIR = DRIVE_ROOT / "artifacts" / OUTPUT_GROUP / DATASET_ID
seed = 2023
phase1_epochs = 90
phase2_epochs = 30
patience = 15
weight_decay = 0.0
training_batch_size = 2048
learning_rate = 0.001
print(torch.__version__, np.__version__, torch.cuda.get_device_name(device))
print("Best weights will be saved under", RUNS_DIR)

## 5. Load the fixed splits and evaluation loaders

Keep existing ID mapping, repeated interactions, history masking, warm full-catalogue ranking and cold-only candidate ranking. Only training interactions enter the graph. Validation selects NDCG@10; final metrics include Recall/NDCG at 10 and 50.

In [ ]:
RECBOLE_DIR.mkdir(parents=True, exist_ok=True)
interaction_columns = ["user_id:token", "item_id:token", "timestamp:float"]
for split_name in ("train", "valid", "test", "cold"):
    split_path = RECBOLE_DIR / f"{DATASET_ID}.{split_name}.inter"
    if not split_path.exists():
        interactions = pd.read_csv(
            PREPARED_DIR / f"{split_name}.tsv.gz",
            sep="\t",
            usecols=interaction_columns,
            dtype={"user_id:token": str, "item_id:token": str},
            float_precision="round_trip",
        )
        # RecBole requires typed headers. Leave source_row out of this export.
        interactions[interaction_columns].to_csv(split_path, sep="\t", index=False)

catalogue_path = RECBOLE_DIR / f"{DATASET_ID}.item"
if not catalogue_path.exists():
    catalogue_ids = pd.read_csv(PREPARED_DIR / "item_ids.tsv.gz", sep="\t", dtype={"item_id": str})
    pd.DataFrame({"item_id:token": catalogue_ids["item_id"]}).to_csv(
        catalogue_path, sep="\t", index=False,
    )

In [ ]:
dataset_settings = {
    "data_path": str(RECBOLE_ROOT),
    "benchmark_filename": ["train", "valid", "test", "cold"],
    "field_separator": "\t",
    "USER_ID_FIELD": "user_id",
    "ITEM_ID_FIELD": "item_id",
    "TIME_FIELD": "timestamp",
    "load_col": {
        "inter": ["user_id", "item_id", "timestamp"],
        "item": ["item_id"],
    },
    "normalize_all": False,
    "use_gpu": False,
}

base_config = Config(model=carec.CollabContex, dataset=DATASET_ID, config_dict=dataset_settings)
# Set before construction: CARec stores graph and item tensors outside registered buffers.
base_config["device"] = device
recbole_dataset = Dataset(base_config)
recbole_product_ids = recbole_dataset.field2id_token["item_id"]
train_dataset, valid_dataset, test_dataset, cold_dataset = recbole_dataset.build()

base_config["metrics"] = ["Recall", "NDCG"]
base_config["topk"] = [10, 50]
base_config["eval_batch_size"] = 64 * train_dataset.item_num

In [ ]:
# Upstream repeatable sampling leaves history empty. Supply cumulative history.
history_sampler = Sampler(
    ["train", "valid", "test", "cold"],
    [train_dataset, valid_dataset, test_dataset, cold_dataset],
)
evaluation_loaders = {}
for split_name, split in [("valid", valid_dataset), ("warm", test_dataset), ("cold", cold_dataset)]:
    phase = "test" if split_name == "warm" else split_name
    evaluation_loaders[split_name] = FullSortEvalDataLoader(
        base_config, split, history_sampler.set_phase(phase), shuffle=False,
    )

cold_product_ids = cold_dataset.inter_feat[base_config["ITEM_ID_FIELD"]].unique()
train_product_ids = train_dataset.inter_feat[base_config["ITEM_ID_FIELD"]].unique()
assert not set(cold_product_ids.tolist()) & set(train_product_ids.tolist())
print({name: len(loader.uid_list) for name, loader in evaluation_loaders.items()})

## 6. Shared model, evaluation and training functions

Cached embeddings remain fixed. The CUDA sparse adjacency sums the original normalized edges, including repeated-edge contributions. Each condition has separate weights and its own Drive folder. Existing phase-two weights trigger evaluation only. Existing phase-one weights trigger only phase two with a fresh optimizer.

Best weights use a temporary file followed by replacement. Drive interruption can still lose the latest write. No optimizer, history or configuration sidecars are written, matching the local weights-only policy.

In [ ]:
def prepare_model(embedding_name):
    embedding_path = EMBEDDING_DIR / embedding_name
    with np.load(embedding_path, allow_pickle=False) as saved:
        item_ids = saved["item_ids"]
        vectors = saved["embeddings"]
        metadata = {
            "embedding_file": str(embedding_path.relative_to(PROJECT_ROOT)),
            "encoder": str(saved["model_id"]), "revision": str(saved["revision"]),
            "prompt": str(saved["prompt"]), "dimension": vectors.shape[1],
        }
    rows = {product_id: row for row, product_id in enumerate(item_ids)}
    assert len(rows) == len(item_ids), "Duplicate cache IDs"
    assert len(vectors) == len(item_ids), "Cache row count differs"
    assert recbole_product_ids[0] == "[PAD]"
    assert set(recbole_product_ids[1:]) == set(item_ids), "Product catalogues differ"
    aligned = np.zeros((train_dataset.item_num, vectors.shape[1]), dtype=np.float32)
    for item_id, product_id in enumerate(recbole_product_ids[1:], start=1):
        aligned[item_id] = vectors[rows[product_id]]

    config = deepcopy(base_config)
    dimension = metadata["dimension"]
    config["seed"] = seed
    config["encoder"] = "LightGCN"
    config["num_layers"] = 2
    config["embedding_size"] = dimension
    config["mlp"] = {
        "input_dim": dimension, "hidden_dims": [dimension, dimension],
        "output_dim": dimension, "num_layers": 2, "dropout": 0.2,
    }
    # Share immutable interactions; give this variant its own item features and metadata.
    model_dataset = copy(train_dataset)
    model_dataset.config = config
    for field in ("field2type", "field2source", "field2seqlen"):
        setattr(model_dataset, field, getattr(train_dataset, field).copy())
    model_dataset.set_field_property("item_emb", FeatureType.FLOAT_SEQ, FeatureSource.ITEM, dimension)
    model_dataset.item_feat = Interaction({
        **train_dataset.item_feat.interaction, "item_emb": torch.from_numpy(aligned),
    })
    init_seed(seed, config["reproducibility"])
    model = carec.CollabContex(config, model_dataset).to(device)
    encoder = model.encoder
    node_count = encoder.n_users + encoder.n_items
    adjacency = torch.sparse_coo_tensor(
        encoder.edge_index.flip(0), encoder.edge_weight, (node_count, node_count),
        device=device,
    ).coalesce().to_sparse_csr()

    def sparse_forward(node_vectors, edge_index, edge_weight):
        return torch.sparse.mm(adjacency, node_vectors)

    encoder.gcn_conv.forward = sparse_forward
    return model, config, metadata

In [ ]:
@torch.no_grad()
def evaluate_carec(model, loader, config, cold_only=False):
    collector = Collector(config)
    item_count = loader.dataset.item_num
    evaluator = Evaluator(config)
    previous_training = model.training
    previous_mlp_bypass = model.encoder.not_apply_mlp
    model.eval()
    if cold_only:
        model.encoder.not_apply_mlp = True
    # full_sort_predict caches graph outputs. Never reuse them across modes or epochs.
    model.restore_user_e = model.restore_item_e = None
    candidate_mask = torch.ones(item_count, dtype=torch.bool, device=config["device"])
    if cold_only:
        candidate_mask[:] = False
        cold_ids = loader.dataset.inter_feat[config["ITEM_ID_FIELD"]].unique()
        candidate_mask[cold_ids.to(config["device"])] = True
    candidate_mask[0] = False

    try:
        for interaction, history, positive_users, positive_items in tqdm(loader, desc="Cold" if cold_only else "Full ranking"):
            scores = model.full_sort_predict(interaction.to(config["device"]))
            scores = scores.view(-1, item_count)
            if not torch.isfinite(scores).all():
                raise ValueError("Non-finite recommendation scores before masking.")
            # Apply the item mask to columns, for every user in the batch.
            scores[:, ~candidate_mask] = -torch.inf
            if history is not None:
                scores[history[0].to(scores.device), history[1].to(scores.device)] = -torch.inf
            collector.eval_batch_collect(scores, interaction, positive_users, positive_items)
        return {name: float(value) for name, value in evaluator.evaluate(collector.get_data_struct()).items()}
    finally:
        model.encoder.not_apply_mlp = previous_mlp_bypass
        model.restore_user_e = model.restore_item_e = None
        model.train(previous_training)

In [ ]:
def train_phase(model, config, phase, max_epochs, run_dir):
    if max_epochs < 1 or patience < 1:
        raise ValueError("Epoch limit and patience must be positive.")
    if phase == 1:
        model.set_item_tutoring_phase()
        uniformity_field = config["USER_ID_FIELD"]
    elif phase == 2:
        model.set_user_tutoring_phase()
        uniformity_field = config["ITEM_ID_FIELD"]
    else:
        raise ValueError("CARec has phases 1 and 2.")

    # Frozen parameters have no optimizer state in this phase.
    model.zero_grad(set_to_none=True)
    optimizer = torch.optim.Adam(
        [parameter for parameter in model.parameters() if parameter.requires_grad],
        lr=learning_rate, weight_decay=weight_decay,
    )
    generator = torch.Generator().manual_seed(seed + phase)
    checkpoint_path = run_dir / f"phase{phase}-best.pt"
    best_score, stale_epochs = -float("inf"), 0
    best_epoch, best_metrics = None, None
    history = []
    started = perf_counter()

    for epoch in range(1, max_epochs + 1):
        model.train()
        indices = torch.randperm(len(train_dataset.inter_feat), generator=generator)
        total_loss, interaction_count = 0.0, 0
        for batch_ids in tqdm(indices.split(training_batch_size), desc=f"Phase {phase}, epoch {epoch}"):
            batch = train_dataset.inter_feat[batch_ids].to(config["device"])
            if batch[uniformity_field].unique().numel() < 2:
                raise ValueError("Uniformity needs at least two distinct active-side IDs.")
            optimizer.zero_grad(set_to_none=True)
            loss = model.calculate_loss(batch)
            if not torch.isfinite(loss):
                raise ValueError("Training loss is not finite.")
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * len(batch)
            interaction_count += len(batch)

        valid_metrics = evaluate_carec(model, evaluation_loaders["valid"], config)
        score = float(valid_metrics["ndcg@10"])
        if not np.isfinite(score):
            raise ValueError("Validation NDCG@10 is not finite.")
        improved = score > best_score
        stale_epochs = 0 if improved else stale_epochs + 1
        history.append({
            "epoch": epoch, "loss": total_loss / interaction_count,
            "valid": valid_metrics, "elapsed_seconds": perf_counter() - started,
        })
        if improved:
            best_score = score
            # Write a temporary file so interruption cannot truncate the previous best.
            temporary = checkpoint_path.with_suffix(".pt.tmp")
            best_epoch, best_metrics = epoch, valid_metrics
            torch.save(
                {name: value.detach().cpu() for name, value in model.state_dict().items()},
                temporary,
            )
            temporary.replace(checkpoint_path)
        print(f"Phase {phase}, epoch {epoch}: loss={history[-1]['loss']:.4f}, valid NDCG@10={score:.4f}, patience={stale_epochs}/{patience}")
        if stale_epochs >= patience:
            break

    # Reload the best weights before the next phase or final evaluation.
    best = torch.load(checkpoint_path, map_location="cpu", weights_only=True)
    model.load_state_dict(best)
    model.restore_user_e = model.restore_item_e = None
    model.eval()
    return {"best_epoch": best_epoch, "valid": best_metrics, "epochs_run": len(history)}

In [ ]:
experiment_results = {}


def run_experiment(embedding_name, *, phase2_only=False):
    variant = Path(embedding_name).stem
    run_dir = RUNS_DIR / variant
    checkpoints = sorted(path for path in run_dir.glob("*.pt") if path.is_file())
    # Seed phase one from the transferred Mac weights, without replacing Drive weights.
    imported_phase1 = PROJECT_ROOT / "artifacts/carec" / DATASET_ID / variant / "phase1-best.pt"
    if not checkpoints and imported_phase1.is_file():
        run_dir.mkdir(parents=True, exist_ok=True)
        temporary = run_dir / "phase1-best.pt.tmp"
        shutil.copyfile(imported_phase1, temporary)
        temporary.replace(run_dir / "phase1-best.pt")
        checkpoints = [run_dir / "phase1-best.pt"]
    phase2_only = not (run_dir / "phase2-best.pt").is_file() and (run_dir / "phase1-best.pt").is_file()
    if phase2_only and not (run_dir / "phase1-best.pt").is_file():
        raise FileNotFoundError(f"Phase 2 requires {run_dir / 'phase1-best.pt'}")
    started = perf_counter()
    model, config, metadata = prepare_model(embedding_name)
    if phase2_only and not (run_dir / "phase2-best.pt").is_file():
        checkpoint_path = run_dir / "phase1-best.pt"
        model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))
        model.restore_user_e = model.restore_item_e = None
        print(f"Phase 2 only: {embedding_name}, loaded {checkpoint_path.name}")
        training = {"phase1": {"best_epoch": None}}
        training["phase2"] = train_phase(model, config, 2, phase2_epochs, run_dir)
    elif checkpoints:
        checkpoint_path = next(
            (run_dir / name for name in ("phase2-best.pt", "phase1-best.pt")
             if (run_dir / name).is_file()),
            None,
        )
        if checkpoint_path is None:
            raise ValueError(f"Checkpoint exists in {run_dir}, but no recognized phase checkpoint was found. Training skipped.")
        model.load_state_dict(torch.load(checkpoint_path, map_location="cpu", weights_only=True))
        if checkpoint_path.name == "phase2-best.pt":
            model.set_user_tutoring_phase()
        else:
            model.set_item_tutoring_phase()
        model.restore_user_e = model.restore_item_e = None
        model.eval()
        print(f"Evaluation only: {embedding_name}, loaded {checkpoint_path.name}")
        training = {"phase1": {"best_epoch": None}, "phase2": {"best_epoch": None}}
    else:
        run_dir.mkdir(parents=True, exist_ok=True)
        # Remove stale sidecars and temporary files before a fresh run.
        old_outputs = (
            "phase1-best.pt", "phase2-best.pt", "phase1-history.json", "phase2-history.json",
            "config.json", "id_maps.npz", "training.json", "results.json", "status.json",
        )
        for filename in old_outputs:
            (run_dir / filename).unlink(missing_ok=True)
            (run_dir / (filename + ".tmp")).unlink(missing_ok=True)
        experiment_results.pop(variant, None)
        print(f"{embedding_name}: {metadata['dimension']} dimensions, model {device}, graph CUDA")
        print(f"Model directory: {run_dir}")
        training = {}
        training["phase1"] = train_phase(model, config, 1, phase1_epochs, run_dir)
        training["phase2"] = train_phase(model, config, 2, phase2_epochs, run_dir)
    test = {
        "warm": evaluate_carec(model, evaluation_loaders["warm"], config),
        "cold": evaluate_carec(model, evaluation_loaders["cold"], config, cold_only=True),
    }
    result = {
        "dimension": metadata["dimension"], "training": training, "test": test,
        "total_seconds": perf_counter() - started, "run_dir": str(run_dir),
    }
    experiment_results[variant] = result
    display(pd.DataFrame(test).T)
    del model
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    return result

## 7. Individual remaining runs

Run cells independently. E5 without instruction starts only phase two. The other three conditions train both phases unless their Drive folders already contain phase weights. A phase-two checkpoint skips all training and evaluates saved weights.

### Multilingual E5-large without instruction, phase 2 only

In [ ]:
e5_large_plain_result = run_experiment("multilingual-e5-large-instruct-without_instruction.npz")

### Multilingual E5-large with instruction

In [ ]:
e5_large_instruct_result = run_experiment("multilingual-e5-large-instruct-with_instruction.npz")

### INSTRUCTOR-base without instruction

In [ ]:
instructor_plain_result = run_experiment("instructor-base-without_instruction.npz")

### INSTRUCTOR-base with instruction

In [ ]:
instructor_instruct_result = run_experiment("instructor-base-with_instruction.npz")

## 8. Results from this session

Save the notebook with its outputs to preserve displayed metrics. Checkpoints remain in Drive after the runtime ends. To bring weights back to the Mac, copy the variant folders from `PML/artifacts/carec-colab/electronics_carec_035_seed2023/` into the corresponding local `artifacts/carec/electronics_carec_035_seed2023/` folders. Preserve any newer local weights.

In [ ]:
result_rows = []
for variant, result in experiment_results.items():
    row = {
        "variant": variant, "dimension": result["dimension"],
        "minutes": result["total_seconds"] / 60,
        "phase1_epoch": result["training"]["phase1"]["best_epoch"],
        "phase2_epoch": result["training"]["phase2"]["best_epoch"],
    }
    for split_name, metrics in result["test"].items():
        row.update({f"{split_name}_{metric}": value for metric, value in metrics.items()})
    result_rows.append(row)
display(pd.DataFrame(result_rows))

## Sources and limits

- [Colab VS Code extension](https://github.com/googlecolab/colab-vscode) documents remote kernel selection and Drive mounting.
- [Colab FAQ](https://research.google.com/colaboratory/faq.html) describes variable GPU availability and session limits.
- [Pinned CARec source](https://github.com/ChenMetanoia/CARec/tree/bbfc7f385e796840f6eb5c967dfc8714284958ce) supplies the model and original loss.
- [PyTorch sparse matrix multiplication](https://docs.pytorch.org/docs/stable/generated/torch.sparse.mm.html) documents the sparse propagation operation.

This notebook was prepared without executing cells, installing dependencies, uploading data or running training. The Linux/CUDA environment, memory fit, runtime and cross-device numerical agreement remain unverified. NumPy 1.26.4 supports Python 3.12; the Mac environment uses NumPy 1.23.5. Models use cached native dimensions, so this is not the later shared-dimension PCA comparison.